# Khảo sát MixLinear — nhánh phụ có ích không, phi tuyến có giúp thêm không

Thiết kế đầy đủ: `docs/THIET_KE_MIXLINEAR_DE_CLAUDE_REVIEW.md`

## Vì sao có khảo sát này

MixLinear-63 đã chạy: `cv_mean 0,672429 ± 0,006570`. Đường hội tụ cho thấy nó
**đã học hết** — epoch 18→19 chỉ còn giảm 0,28%, phẳng hơn cả TCN-64 — nhưng
dừng ở `train_mse 0,0500`, gấp 3,8 lần TCN. Tức là chạm giới hạn **sức chứa**,
không phải thiếu epoch.

Mà MixLinear gần như kịch trần thiết kế của chính nó. Hai lớp `FLinear` nối
nhau không có phi tuyến nên hợp lại chỉ là một ma trận, hạng tối đa
`min(lpf, mix_hidden, 3)`:

| `mix_hidden` | tham số | hạng thật |
|---:|---:|---:|
| 2 (đang dùng) | 63 | 2 |
| **3** | **79** | **3 — kịch trần** |
| 8 | 159 | 3 |
| 32 | 543 | 3 |

Từ 3 trở lên, thêm tham số **không thể** thêm khả năng biểu diễn. Muốn có sức
chứa thật thì phải thêm đường đi mới.

## Thiết kế giai thừa

|  | không nhánh phụ | có nhánh phụ |
|---|---|---|
| **không MixLinear** | — | **C0** (929) |
| **có MixLinear** | **B0** (63, đã chạy) | **C2** (992) |

Thêm **C3** (992): giống hệt C2, chỉ khác một hàm `GELU` giữa hai lớp nhánh phụ.

Bốn phép so đọc được:

| so sánh | trả lời câu gì | sạch không |
|---|---|---|
| **C3 − C2** | phi tuyến giúp gì | **sạch** — cùng 992 tham số, khác đúng GELU |
| **C2 − C0** | thêm MixLinear khi đã có nhánh phụ | lệch 63 tham số |
| **C2 − B0** | thêm nhánh phụ khi đã có MixLinear | lệch 929 tham số |
| C0 − B0 | 929 tham số tuyến tính so với 63 tham số MixLinear | đổi cả hai biến |

## Ba cấu hình chạy trong notebook này

| | model | nhánh phụ | tham số |
|---|---|---|---:|
| **C0** | `low_rank_linear` | `Linear(200,4) → Linear(4,25)`, **không** có MixLinear | 929 |
| **C2** | `mix_linear_linear` | MixLinear + nhánh trên | 992 |
| **C3** | `mix_linear_mlp` | MixLinear + nhánh trên **có GELU** | 992 |

Nhánh phụ nhận `x − mean(x)`, còn MixLinear tự trừ rồi cộng lại bên trong, nên
trung bình được cộng đúng **một** lần. Nhánh phụ nhìn dữ liệu ở thang biên độ
gốc, không chia độ lệch chuẩn.

Ghép là **phép cộng thuần**: không hệ số trộn học được, không gate, không norm,
không dropout. Train cả nền lẫn nhánh phụ cùng lúc từ đầu, không đóng băng gì.

## Ba điều phải ghi khi báo cáo

**1. Nhánh phụ là đề xuất của đồ án, không phải MixLinear nguyên bản.** LSTNet
(SIGIR 2018) và N-BEATS (ICLR 2020) là tiền lệ cho việc ghép thành phần tuyến
tính với phi tuyến, **không** phải bằng chứng cho cấu hình chiều 4 trên UWB.
Chiều 4 là lựa chọn thử nghiệm theo ngân sách, chưa được chứng minh tối ưu.

**2. C2 và C3 không thật sự bằng nhau tuyệt đối.** Ở C2, `bias` lớp đầu bị hấp
thụ vào `bias` lớp sau, vì hai `Linear` không phi tuyến hợp lại thành đúng một
phép affine. Nên C2 có **988** tham số tác dụng độc lập so với 992 của C3.
Chênh 0,4%, không hỏng phép so, nhưng đừng viết "cùng hệt số tham số".

**3. Một seed là sàng lọc, không phải kiểm định.** B0 có `seed_std` **0,0066**,
ba seed trải 0,6648–0,6764. Nên chênh lệch dưới khoảng **0,007** trên một seed
là ngẫu nhiên, không đọc được gì. Chỉ khi chênh lớn hơn thế mới đáng chạy tiếp
seed 1 và 2.

Và vì train chung từ đầu, **không** được nói nhánh phụ "học phần sai của
MixLinear" — nó chỉ là một đường đi song song, hai bên học cùng nhau.

## 1. Chuẩn bị Colab

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn rồi vào thư mục đó.

In [ ]:
# Xoá trước để chạy lại ô này luôn lấy mã mới nhất, không dính bản cũ.
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

## 2. Kiểm ba bản cài đặt

Chín phép kiểm mỗi cấu hình. Bốn phép nhắm đúng những chỗ dễ sai của thiết kế
ghép nhánh:

    mục 6   gradient tới CẢ nền lẫn nhánh phụ — nếu một bên nằm chết thì
            kết quả nói về kiến trúc khác hẳn kiến trúc mình tưởng
    mục 7   xoá trọng số nhánh phụ, đầu ra phải khớp ĐÚNG MixLinear nền —
            chứng minh ghép là cộng thuần, không có gate hay hệ số ẩn
    mục 8   đưa vào chuỗi có trung bình 50, đầu ra phải quanh 50 chứ không
            phải 100 — trung bình chỉ được cộng một lần
    mục 9   GELU thật sự đổi giá trị, Identity thật sự giữ nguyên

Với C0 thì mục 6 xác nhận hai lớp không phi tuyến hợp lại chỉ là một ma trận
hạng tối đa 4, và in kèm số tham số của `Linear(200, 25)` đầy đủ để thấy phép
tách làm hai lớp tiết kiệm bao nhiêu.

In [ ]:
!python scripts/check_model.py --model low_rank_linear
!python scripts/check_model.py --model mix_linear_linear
!python scripts/check_model.py --model mix_linear_mlp

Xác nhận thêm một điều mà `check_model` chạy riêng lẻ không thấy được:
**C2 và C3 phải có cùng trọng số khởi tạo** khi cùng seed. Nếu khác thì phép so
C3 − C2 lẫn cả chênh lệch khởi tạo, không còn cô lập được GELU.

`Identity` và `GELU` đều không có tham số và không tiêu thụ số ngẫu nhiên, nên
chỉ cần dựng module đúng thứ tự là được.

In [ ]:
import torch; from src import models, training
training.set_seed(0); d2 = models.build_model("mix_linear_linear").state_dict()
training.set_seed(0); d3 = models.build_model("mix_linear_mlp").state_dict()
print("cùng trọng số khởi tạo:", all(torch.equal(d2[k], d3[k]) for k in d2))

## 3. Chạy ba cấu hình, mỗi cấu hình MỘT seed

Giao thức giữ nguyên: 20 epoch, Adam lr 1e-4, batch 64, MSE, `corr` 0,9, bốn
fold cũ. Chỉ đổi kiến trúc.

Nhóm kết quả `tn_mixlinear_ablation`, thư mục `runs/tn_mixlinear_ablation/`
riêng — không lẫn vào bảng TN1. Không gán số TN vì `docs/PROTOCOL.md` đang dùng
TN4 cho ngưỡng `corr`.

Đo từ lần chạy B0: 4,6 phút mỗi fold train, cộng khoảng 25 phút chấm điểm mỗi
seed. Ba cấu hình khoảng **2,2 giờ**.

In [ ]:
!python scripts/run_cv.py --experiment tn_mixlinear_ablation --model low_rank_linear --seed 0
!python scripts/run_cv.py --experiment tn_mixlinear_ablation --model mix_linear_linear --seed 0
!python scripts/run_cv.py --experiment tn_mixlinear_ablation --model mix_linear_mlp --seed 0

## 4. Cất kết quả

In [ ]:
!python scripts/save_results.py tn_mixlinear_ablation --out tn_mixlinear_ablation

## 5. Đường hội tụ của cả ba

Cùng câu hỏi đã hỏi cho B0: điểm thấp là do hết sức chứa, hay do 20 epoch chưa
đủ? B0 dừng ở `train_mse` **0,0500** và đã phẳng. Ba cấu hình này nhiều tham số
hơn nên đáng lẽ phải xuống thấp hơn — nếu không thì sức chứa thêm vào không
được dùng.

In [ ]:
!tail -3 runs/tn_mixlinear_ablation/low_rank_linear_h4_mse_corr0.9_seed0_val_AB/curve.csv
!tail -3 runs/tn_mixlinear_ablation/mix_linear_linear_p10_lpf5_h4_mse_corr0.9_seed0_val_AB/curve.csv
!tail -3 runs/tn_mixlinear_ablation/mix_linear_mlp_p10_lpf5_h4_mse_corr0.9_seed0_val_AB/curve.csv

## 6. Bảng so

Chỉ thấy ba cấu hình của phiên này. Mốc để đặt cạnh:

| | tham số | cv_mean |
|---|---:|---:|
| B0 MixLinear, seed 0 | 63 | **0,6760** |
| B0 MixLinear, 3 seed | 63 | 0,6724 ± 0,0066 |
| LSTM-67 | 56.908 | 0,7532 |

**So với `0,6760` của B0 seed 0**, không so với trung bình ba seed — một seed
với một seed. Và nhớ ngưỡng nhiễu 0,007.

In [ ]:
!python scripts/compare_cv.py --experiment tn_mixlinear_ablation

## 7. Ngắt phiên

In [ ]:
from google.colab import runtime
runtime.unassign()